In [13]:
!pip install folium

In [14]:
!pip install gpxpy

In [21]:
import folium 
import gpxpy
import os
import json
from branca.element import MacroElement, Template

In [22]:
m = folium.Map(location=[44.98670, 20.21098], zoom_start=9)

In [23]:
class ClickableGPXLine(MacroElement):
    """
    Adds Leaflet JS behavior:
    - click line once: highlight it and open popup
    - click same line again: restore original color and close popup
    - click another line: restore previous line, highlight new line
    """

    _template = Template("""
    {% macro script(this, kwargs) %}
    (function() {
        var map = {{ this._parent.get_name() }};
        var layer = {{ this.layer_name }};

        var normalStyle = {{ this.normal_style }};
        var clickedStyle = {{ this.clicked_style }};

        if (!map._activeGPXLineState) {
            map._activeGPXLineState = {
                activeLayer: null
            };
        }

        layer._normalGPXStyle = normalStyle;

        layer.on('click', function(e) {
            var state = map._activeGPXLineState;

            // If another line is active, reset it first
            if (state.activeLayer && state.activeLayer !== layer) {
                state.activeLayer.setStyle(state.activeLayer._normalGPXStyle);
                state.activeLayer.closePopup();
            }

            // If this same line is clicked again, toggle it off
            if (state.activeLayer === layer) {
                layer.setStyle(normalStyle);
                layer.closePopup();
                state.activeLayer = null;
            } else {
                layer.setStyle(clickedStyle);
                layer.openPopup(e.latlng);
                state.activeLayer = layer;
            }
        });
    })();
    {% endmacro %}
    """)

    def __init__(self, layer, normal_style, clicked_style):
        super().__init__()
        self.layer_name = layer.get_name()
        self.normal_style = json.dumps(normal_style)
        self.clicked_style = json.dumps(clicked_style)

In [24]:
def overlayGPX(
    gpxData,
    fmap,
    color="blue",
    weight=4,
    popup_text="GPX route",
    clicked_color="red",
    clicked_weight=None,
):
    """
    Overlay a clickable GPX route on a Folium map.

    Parameters
    ----------
    gpxData : str
        Path to GPX file.
    fmap : folium.Map
        Existing Folium map.
    color : str
        Normal line color.
    weight : int or float
        Normal line weight.
    popup_text : str
        Text shown when the line is clicked.
    clicked_color : str
        Color used when the line is selected.
    clicked_weight : int or float, optional
        Weight used when selected. Defaults to original weight.
    """

    with open(gpxData, "r", encoding="utf-8") as gpx_file:
        gpx = gpxpy.parse(gpx_file)

    points = []

    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append((point.latitude, point.longitude))

    if not points:
        raise ValueError(f"No track points found in GPX file: {gpxData}")

    normal_style = {
        "color": color,
        "weight": weight,
        "opacity": 1,
    }

    clicked_style = {
        "color": clicked_color,
        "weight": clicked_weight if clicked_weight is not None else weight,
        "opacity": 1,
    }

    line = folium.PolyLine(
        points,
        color=color,
        weight=weight,
        opacity=1,
    )
    
    popup_html = f"""
    <div style="
        display: inline-block;
        width: max-content;
        min-width: 160px;
        max-width: 600px;
        white-space: nowrap;
    ">
        {popup_text}
    </div>
    """
    
    line.add_child(
        folium.Popup(
            popup_html,
            max_width=650
        )
    )
    
    line.add_to(fmap)

    click_handler = ClickableGPXLine(
        layer=line,
        normal_style=normal_style,
        clicked_style=clicked_style,
    )

    fmap.add_child(click_handler)

    return line

In [25]:
overlayGPX("trips/2024-03-29-Dragova-greda.gpx", m, "darkred", 3, "<b>Dragova greda</b><br>March 29th, 2024<br><i>Biking, distance 30 km, elevation gain 40 m</i>", "darkred", 10)
overlayGPX("trips/2024-03-30-Kovilovo.gpx", m, "darkred", 3, "<b>Kovilovo</b><br>March 30th, 2024<br><i>Biking, distance 55 km, elevation gain 240 m</i>", "darkred", 10)
overlayGPX("trips/2024-04-06-Slanci-Vinča.gpx", m, "darkred", 3, "<b>Slanci i Vinča</b><br>April 6th, 2024<br><i>Biking, distance 54 km, elevation gain 720 m</i>", "darkred", 10)
overlayGPX("trips/2024-04-13-Manastir-Rakovica.gpx", m, "darkred", 3, "<b>Manastir Rakovica</b><br>April 13th, 2024<br><i>Biking, distance 58 km, elevation gain 690 m</i>", "darkred", 10)
overlayGPX("trips/2024-05-01-Ada.gpx", m, "darkred", 3, "<b>Ada</b><br>May 1st, 2024<br><i>Biking, distance 23 km, elevation gain 100 m</i>", "darkred", 10)
overlayGPX("trips/2024-05-04-Jakovo.gpx", m, "darkred", 3, "<b>Jakovo</b><br>May 4th, 2024<br><i>Biking, distance 52 km, elevation gain 190 m</i>", "darkred", 10)
overlayGPX("trips/2024-05-11-Manastiri-Svetog-Hristofora-i-Fenek.gpx", m, "darkred", 3, "<b>Manastiri Svetog Hristofora i Fenek</b><br>May 11th, 2024<br><i>Biking, distance 101 km, elevation gain 550 m</i>", "darkred", 10)
overlayGPX("trips/2024-05-25-Manastir-Svetog-Justina-Ćelijskog.gpx", m, "darkred", 3, "<b>Manastir Svetog Justina Ćelijskog</b><br>May 25th, 2024<br><i>Biking, distance 96 km, elevation gain 1080 m</i>", "darkred", 10)
overlayGPX("trips/2024-06-02-Smederevo.gpx", m, "darkred", 3, "<b>Smederevo</b><br>June 2nd, 2024<br><i>Biking, distance 141 km, elevation gain 860 m</i>", "darkred", 10)
overlayGPX("trips/2024-06-16-Manastir-Presvete-Bogorodice-Trojeručice-i-Manastir-Rajinovac.gpx", m, "darkred", 3, "<b>Manastiri Presvete Bogorodice Trojeručice i Rajinovac</b><br>June 16th, 2024<br><i>Biking, distance 62 km, elevation gain 1070 m</i>", "darkred", 10)
overlayGPX("trips/2024-06-22-Inđija.gpx", m, "darkred", 3, "<b>Inđija</b><br>June 22nd, 2024<br><i>Biking, distance 133 km, elevation gain 690 m</i>", "darkred", 10)
overlayGPX("trips/2024-08-03-Pančevo.gpx", m, "darkred", 3, "<b>Pančevo</b><br>August 3rd, 2024<br><i>Biking, distance 49 km, elevation gain 180 m</i>", "darkred", 10)
overlayGPX("trips/2024-08-10-Deliblatska-peščara.gpx", m, "darkred", 3, "<b>Deliblatska peščara</b><br>August 10th, 2024<br><i>Biking, distance 175 km, elevation gain 860 m</i>", "darkred", 10)
overlayGPX("trips/2024-09-13-Vladimir.gpx", m, "darkred", 3, "<b>Meeting Vovak</b><br>September 13th, 2024<br><i>Biking, distance 92 km, elevation gain 470 m</i>", "darkred", 10)
overlayGPX("trips/2024-10-12-Višnjica.gpx", m, "darkred", 3, "<b>Višnjica</b><br>October 12th, 2024<br><i>Biking, distance 60 km, elevation gain 510 m</i>", "darkred", 10)
overlayGPX("trips/2024-10-13-Dobanovci.gpx", m, "darkred", 3, "<b>Dobanovci</b><br>October 13th, 2024<br><i>Biking, distance 101 km, elevation gain 310 m</i>", "darkred", 10)
overlayGPX("trips/2024-10-20-Novo-Bežanijsko-Groblje.gpx", m, "darkred", 3, "<b>Novo Bežanijsko groblje</b><br>October 20th, 2024<br><i>Biking, distance 35 km, elevation gain 200 m</i>", "darkred", 10)
overlayGPX("trips/2025-04-18-Sremačka-šuma.gpx", m, "darkred", 3, "<b>Sremačka šuma</b><br>April 18th, 2025<br><i>Biking, distance 47 km, elevation gain 630 m</i>", "darkred", 10)
overlayGPX("trips/2025-04-26-Radiofar.gpx", m, "darkred", 3, "<b>Radiofar</b><br>April 26th, 2025<br><i>Biking, distance 35 km, elevation gain 210 m</i>", "darkred", 10)
overlayGPX("trips/2025-05-01-Veliko-Blato.gpx", m, "darkred", 3, "<b>Veliko Blato</b><br>May 1st, 2025<br><i>Biking, distance 70 km, elevation gain 280 m</i>", "darkred", 10)
overlayGPX("trips/2025-05-10-Zemun-Polje.gpx", m, "darkred", 3, "<b>Zemun Polje</b><br>May 10th, 2025<br><i>Biking, distance 54 km, elevation gain 260 m</i>", "darkred", 10)
overlayGPX("trips/2025-05-17-Ruski-nekropol.gpx", m, "darkred", 3, "<b>Ruski nekropol</b><br>May 17th, 2025<br><i>Biking, distance 41 km, elevation gain 590 m</i>", "darkred", 10)
overlayGPX("trips/2025-05-24-Hipodrom.gpx", m, "darkred", 3, "<b>Hipodrom</b><br>May 24th, 2025<br><i>Biking, distance 47 km, elevation gain 670 m</i>", "darkred", 10)
overlayGPX("trips/2025-06-14-Pavel.gpx", m, "darkred", 3, "<b>Driving with Pavel</b><br>June 14th, 2025<br><i>Biking, distance 51 km, elevation gain 420 m</i>", "darkred", 10)
overlayGPX("trips/2025-06-28-Blok-67.gpx", m, "darkred", 3, "<b>Blok 67</b><br>June 28th, 2025<br><i>Biking, distance 16 km, elevation gain 180 m</i>", "darkred", 10)
overlayGPX("trips/2025-07-12-Ivanovo.gpx", m, "darkred", 3, "<b>Ivanovo</b><br>July 12th, 2025<br><i>Biking, distance 84 km, elevation gain 330 m</i>", "darkred", 10)
overlayGPX("trips/2025-07-19-Šimanovci.gpx", m, "darkred", 3, "<b>Šimanovci</b><br>July 19th, 2025<br><i>Biking, distance 91 km, elevation gain 290 m</i>", "darkred", 10)
overlayGPX("trips/2025-08-03-Ada.gpx", m, "darkred", 3, "<b>Ada</b><br>August 3rd, 2025<br><i>Biking, distance 21 km, elevation gain 130 m</i>", "darkred", 10)
overlayGPX("trips/2025-08-23-Obrenovac.gpx", m, "darkred", 3, "<b>Obrenovac</b><br>August 23rd, 2025<br><i>Biking, distance 102 km, elevation gain 350 m</i>", "darkred", 10)
overlayGPX("trips/2025-09-06-Manastir-Fenek.gpx", m, "darkred", 3, "<b>Manastir Fenek</b><br>September 6th, 2025<br><i>Biking, distance 59 km, elevation gain 240 m</i>", "darkred", 10)
overlayGPX("trips/2025-10-11-Kraljeva-česma.gpx", m, "darkred", 3, "<b>Kraljeva česma</b><br>October 11th, 2025<br><i>Biking, distance 68 km, elevation gain 1080 m</i>", "darkred", 10)
overlayGPX("trips/2026-03-16-Sonya.gpx", m, "darkred", 3, "<b>Driving with Sonya</b><br>March 16th, 2026<br><i>Biking, driving 34 km, elevation gain 160 m</i>", "darkred", 10)
overlayGPX("trips/2026-04-04-Bačka-Palanka.gpx", m, "darkred", 3, "<b>Bačka Palanka</b><br>April 4th, 2026<br><i>Biking, distance 97 km, elevation gain 260 m</i>", "darkred", 10)
overlayGPX("trips/2026-04-06-Petrovaradin.gpx", m, "darkred", 3, "<b>Petrovaradin</b><br>April 6th, 2026<br><i>Biking, distance 16 km, elevation gain 90 m</i>", "darkred", 10)
overlayGPX("trips/2026-05-09-Belgrade-Novi-Sad.gpx", m, "darkred", 3, "<b>Belgrade — Novi Sad</b><br>May 9th, 2026<br><i>Biking, distance 103 km, elevation gain 730 m</i>", "darkred", 10)
overlayGPX("trips/2026-05-10-Novi-Sad-Belgrade.gpx", m, "darkred", 3, "<b>Novi Sad — Belgrade</b><br>May 10th, 2026<br><i>Biking, distance 87 km, elevation gain 630 m</i>", "darkred", 10)

overlayGPX("trips/2023-07-08-Manastir-Slanci.gpx", m, "darkblue", 3, "<b>Manastir Slanci</b><br>July 8th, 2023<br><i>Walking, distance 38 km, elevation 550 m</i>", "darkblue", 10)
overlayGPX("trips/2024-05-05-Manastir-Rakovica.gpx", m, "darkblue", 3, "<b>Manastir Rakovica</b><br>May 5th, 2024<br><i>Walking, distance 19 km, elevation gain 400 m</i>", "darkblue", 10)
overlayGPX("trips/2024-06-08-Veliko-ratno-ostrvo.gpx", m, "darkblue", 3, "<b>Veliko Ratno ostrvo</b><br>June 8th, 2024<br><i>Walking, distance 26 km, elevation gain 180 m</i>", "darkblue", 10)
overlayGPX("trips/2024-12-30-Avala.gpx", m, "darkblue", 3, "<b>Avala</b><br>December 30th, 2024<br><i>Walking, distance 19 km, elevation gain 580 m</i>", "darkblue", 10)
overlayGPX("trips/2026-05-02-Manastiri-Fruske-1-3.gpx", m, "darkblue", 3, "<b>Manastiri Grgeteg, Velika Remeta i Krušedol</b><br>May 2nd, 2026<br><i>Walking, distance 22 km, elevation gain 620 m</i>", "darkblue", 10)

In [26]:
def dynamic_width_popup(html, min_width=80, max_width=600):
    popup_html = f"""
    <div style="
        display: inline-block;
        width: max-content;
        min-width: {min_width}px;
        max-width: {max_width}px;
        white-space: nowrap;
    ">
        {html}
    </div>
    """

    return folium.Popup(
        popup_html,
        max_width=max_width + 50
    )

color_manastir = "red"
icon_manastir = "church"
z_index_manastir = 2000

color_forest = "green"
icon_forest = "tree"
z_index_nature = 1000

color_mountain = "green"
icon_mountain = "mountain"

color_water = "green"
icon_water = "water"

color_travel = "blue"
icon_travel = "home"
z_index_travel = 100

In [27]:
folium.Marker(
    [44.808512, 20.571532], 
    popup=dynamic_width_popup("<b>Slanci</b><br>July 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.908259, 20.422440], 
    popup=dynamic_width_popup("<b>Kovilovo</b><br>March 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.875382, 20.450733], 
    popup=dynamic_width_popup("<b>Borča</b><br>March 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.810701, 20.597885], 
    popup=dynamic_width_popup("<b>Veliko Selo</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.762125, 20.623390], 
    popup=dynamic_width_popup("<b>Vinča</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.731436, 20.575265], 
    popup=dynamic_width_popup("<b>Leštane</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.754102, 20.260131], 
    popup=dynamic_width_popup("<b>Jakovo</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.790994, 20.270136], 
    popup=dynamic_width_popup("<b>Surčin</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.730052, 20.317110], 
    popup=dynamic_width_popup("<b>Ostružnica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.678686, 20.304759], 
    popup=dynamic_width_popup("<b>Umka</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.660864, 20.310115], 
    popup=dynamic_width_popup("<b>Rucka</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.652332, 20.260206], 
    popup=dynamic_width_popup("<b>Barič</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.678620, 20.202372], 
    popup=dynamic_width_popup("<b>Zabrežje</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67076, 20.16848],
    popup=dynamic_width_popup("<b>Urovci</b><br>May 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.722294, 20.228663], 
    popup=dynamic_width_popup("<b>Boljevci</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.664115, 20.353923], 
    popup=dynamic_width_popup("<b>Velika Moštanica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.623750, 20.370699], 
    popup=dynamic_width_popup("<b>Meljak</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.601047, 20.400301], 
    popup=dynamic_width_popup("<b>Guncati</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.578149, 20.415967], 
    popup=dynamic_width_popup("<b>Barajevo</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.630595, 20.432750], 
    popup=dynamic_width_popup("<b>Stara Lipovica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.682874, 20.436493], 
    popup=dynamic_width_popup("<b>Rušanj</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.694958, 20.489928], 
    popup=dynamic_width_popup("<b>Pinosava</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.705059, 20.511254], 
    popup=dynamic_width_popup("<b>Beli Potok</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87111, 20.64117], 
    popup=dynamic_width_popup("<b>Pančevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.80808, 20.70597], 
    popup=dynamic_width_popup("<b>Starčevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.75823, 20.73683], 
    popup=dynamic_width_popup("<b>Omoljica</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.81787, 20.87654], 
    popup=dynamic_width_popup("<b>Bavanište</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.74222, 20.97733], 
    popup=dynamic_width_popup("<b>Kovin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.74222, 20.97733], 
    popup=dynamic_width_popup("<b>Kovin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66501, 20.92724], 
    popup=dynamic_width_popup("<b>Smederevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66501, 20.92724], 
    popup=dynamic_width_popup("<b>Smederevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.65414, 20.825447], 
    popup=dynamic_width_popup("<b>Seone</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.64770, 20.75849], 
    popup=dynamic_width_popup("<b>Brestovik</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67191, 20.72026], 
    popup=dynamic_width_popup("<b>Grocka</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.6953, 20.67443], 
    popup=dynamic_width_popup("<b>Zaklopača</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.73565, 20.60817], 
    popup=dynamic_width_popup("<b>Boleč</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66478, 20.59215], 
    popup=dynamic_width_popup("<b>Vrčin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.62510, 20.58102], 
    popup=dynamic_width_popup("<b>Ripanj</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.59294, 20.61369], 
    popup=dynamic_width_popup("<b>Mala Ivanča</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.56871, 20.64705], 
    popup=dynamic_width_popup("<b>Mali Požarevac</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.58495, 20.68899], 
    popup=dynamic_width_popup("<b>Drazanj</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.58346, 20.73556], 
    popup=dynamic_width_popup("<b>Umčari</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.61081, 20.72387], 
    popup=dynamic_width_popup("<b>Pudarci</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.63372, 20.69832], 
    popup=dynamic_width_popup("<b>Begaljica</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.94540, 20.21771], 
    popup=dynamic_width_popup("<b>Nova Pazova</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.984613, 20.158889],
    popup=dynamic_width_popup("<b>Stara Pazova</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.04787, 20.08061],
    popup=dynamic_width_popup("<b>Inđija</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.08112, 20.17692],
    popup=dynamic_width_popup("<b>Novi Karlovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.12733, 20.24097],
    popup=dynamic_width_popup("<b>Novi Slankamen</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.14137, 20.25791],
    popup=dynamic_width_popup("<b>Stari Slankamen</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.07113, 20.32517],
    popup=dynamic_width_popup("<b>Surduk</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.01921, 20.33316],
    popup=dynamic_width_popup("<b>Belegiš</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.98215, 20.28114],
    popup=dynamic_width_popup("<b>Stari Banovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.95076, 20.28270],
    popup=dynamic_width_popup("<b>Novi Banovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.90083, 20.87698],
    popup=dynamic_width_popup("<b>Dolovo</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87947, 20.97341],
    popup=dynamic_width_popup("<b>Mramorak</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.83981, 21.04285],
    popup=dynamic_width_popup("<b>Deliblato</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.94112, 21.12513],
    popup=dynamic_width_popup("<b>Šušara</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.02173, 21.18425],
    popup=dynamic_width_popup("<b>Izbište</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.04306, 21.15672],
    popup=dynamic_width_popup("<b>Uljma</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.05270, 21.06952],
    popup=dynamic_width_popup("<b>Nikolinci</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.05107, 21.01602],
    popup=dynamic_width_popup("<b>Banatski Karlovac</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.03202, 20.86497],
    popup=dynamic_width_popup("<b>Vladimirovac</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.99002, 20.78345],
    popup=dynamic_width_popup("<b>Banatsko Novo Selo</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87102, 20.20157],
    popup=dynamic_width_popup("<b>Ugrinovci</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.85519, 20.15667],
    popup=dynamic_width_popup("<b>Grmovac</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.82765, 20.22290],
    popup=dynamic_width_popup("<b>Dobanovci</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.82961, 20.31408],
    popup=dynamic_width_popup("<b>Radiofar</b><br>April 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.88865, 20.53643],
    popup=dynamic_width_popup("<b>Ovča</b><br>May 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.73999, 20.69607],
    popup=dynamic_width_popup("<b>Ivanovo</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.93559, 20.15159],
    popup=dynamic_width_popup("<b>Vojka</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.88942, 20.13837],
    popup=dynamic_width_popup("<b>Krnješevci</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87330, 20.09689],
    popup=dynamic_width_popup("<b>Šimanovci</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.83590, 20.11464],
    popup=dynamic_width_popup("<b>Deč</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.79360, 20.15139],
    popup=dynamic_width_popup("<b>Petrovčić</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.78067, 20.20291],
    popup=dynamic_width_popup("<b>Bečmen</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.65469, 20.20094],
    popup=dynamic_width_popup("<b>Obrenovac</b><br>August 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.65813, 20.20000],
    popup=dynamic_width_popup("<b>Rvati</b><br>August 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67779, 20.53693],
    popup=dynamic_width_popup("<b>Šuplja Stena</b><br>October 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.69878, 20.55698],
    popup=dynamic_width_popup("<b>Zuce</b><br>October 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.23938, 19.88556],
    popup=dynamic_width_popup("<b>Petrovaradin</b><br>April 2026"),
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.25561, 19.84568], 
    popup=dynamic_width_popup("<b>Novi Sad</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24953, 19.75643], 
    popup=dynamic_width_popup("<b>Veternik</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24153, 19.71337], 
    popup=dynamic_width_popup("<b>Futog</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24116, 19.62194], 
    popup=dynamic_width_popup("<b>Begeč</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.27614, 19.56327], 
    popup=dynamic_width_popup("<b>Gložan</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.26758, 19.53162], 
    popup=dynamic_width_popup("<b>Čelarevo</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.25000, 19.38449], 
    popup=dynamic_width_popup("<b>Bačka Palanka</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.19404, 19.89414], 
    popup=dynamic_width_popup("<b>Bukovac</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.13519, 19.90252], 
    popup=dynamic_width_popup("<b>Grgeteg</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.14630, 19.92400], 
    popup=dynamic_width_popup("<b>Velika Remeta</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.12076, 19.93840], 
    popup=dynamic_width_popup("<b>Krušedol Prnjavor</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.17379, 20.18699], 
    popup=dynamic_width_popup("<b>Slankamenački Vinogradi</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.16013, 20.12288], 
    popup=dynamic_width_popup("<b>Krčedin</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.15683, 20.06954], 
    popup=dynamic_width_popup("<b>Beška</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.16133, 20.02282], 
    popup=dynamic_width_popup("<b>Čortanovci</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.20296, 19.93396], 
    popup=dynamic_width_popup("<b>Sremski Karlovci</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.11390, 19.94675], 
    popup=dynamic_width_popup("<b>Krušedol Selo</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.10177, 19.99386], 
    popup=dynamic_width_popup("<b>Maradik</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)



In [28]:
folium.Marker(
    [44.795907, 20.584343], 
    popup=dynamic_width_popup("<b>Manastir Arhiđakona Stefana</b><br>July 2023"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.777055, 20.609738], 
    popup=dynamic_width_popup("<b>Manastir Vinča (skoro u izgradnji)</b><br>April 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.733593, 20.551392], 
    popup=dynamic_width_popup("<b>Manastir Svetog Đorđa</b><br>April 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.730610, 20.447130], 
    popup=dynamic_width_popup("<b>Manastir Rakovica</b><br>April 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.641417, 20.240282], 
    popup=dynamic_width_popup("<b>Manastir Svetog Hristofora</b><br>May 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.739699, 20.224508], 
    popup=dynamic_width_popup("<b>Manastir Fenek</b><br>May 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.790633, 20.443814], 
    popup=dynamic_width_popup("<b>Manastir Vavedenje Presvete Bogorodice u Beogradu</b><br>May 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.617262, 20.431781], 
    popup=dynamic_width_popup("<b>Manastir Svetog Justina Ćelijskog</b><br>May 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.82811, 20.68515], 
    popup=dynamic_width_popup("<b>Manastir Vojlovica</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.82712, 20.89400], 
    popup=dynamic_width_popup("<b>Manastir Bavanište</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.84127, 20.40983], 
    popup=dynamic_width_popup("<b>Manastir Svetog arhangela Gavrila</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.79683, 20.49050], 
    popup=dynamic_width_popup("<b>Manastir Svetog Antuna Padovanskog</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.62855, 20.58999], 
    popup=dynamic_width_popup("<b>Manastir Svetog Georgija SSPC</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.61645, 20.58874], 
    popup=dynamic_width_popup("<b>Manastir Presvete Bogorodice Trojeručice</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [44.62486, 20.71339], 
    popup=dynamic_width_popup("<b>Manastir Rajinovac</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.06216, 20.14805], 
    popup=dynamic_width_popup("<b>Manastir Svetog Marka</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.06539, 20.31390], 
    popup=dynamic_width_popup("<b>Manastir Svetog Novomučenika Haritona Kosovskog</b><br>June 2024"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.13801, 19.90201], 
    popup=dynamic_width_popup("<b>Manastir Grgeteg</b><br>May 2026"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.14279, 19.91771], 
    popup=dynamic_width_popup("<b>Manastir Velika Remeta</b><br>May 2026"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.14279, 19.91771], 
    popup=dynamic_width_popup("<b>Manastir Velika Remeta</b><br>May 2026"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.11944, 19.94003], 
    popup=dynamic_width_popup("<b>Manastir Krušedol</b><br>May 2026"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)

folium.Marker(
    [45.20410, 19.93064], 
    popup=dynamic_width_popup("<b>Manastir Vavedenja Presvete Bogorodice</b><br>May 2026"), 
    icon=folium.Icon(color=color_manastir, icon=icon_manastir, prefix="fa"), z_index_offset=z_index_manastir
).add_to(m)




In [29]:
folium.Marker(
    [44.80328, 20.51302], 
    popup=dynamic_width_popup("<b>Zvezdarska šuma</b><br>July 2022"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.78838, 20.39298], 
    popup=dynamic_width_popup("<b>Ada Ciganlija</b><br>August 2023"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.77901, 20.43938], 
    popup=dynamic_width_popup("<b>Topčider</b><br>April 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.74154, 20.45434], 
    popup=dynamic_width_popup("<b>Miljakovačka šuma</b><br>April 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.76263, 20.43589], 
    popup=dynamic_width_popup("<b>Šuma Košutnjak</b><br>April 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.70167, 20.34511], 
    popup=dynamic_width_popup("<b>Ostružnička šuma</b><br>May 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.65718, 20.30111], 
    popup=dynamic_width_popup("<b>Sujetsko i Žuta brda</b><br>May 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.66559, 20.23306], 
    popup=dynamic_width_popup("<b>Izletište zabran</b><br>May 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.83127, 20.43761], 
    popup=dynamic_width_popup("<b>Veliko Ratno ostrvo</b><br>June 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.89958, 21.10282], 
    popup=dynamic_width_popup("<b>Deliblatska peščara</b><br>August 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.81960, 20.16865], 
    popup=dynamic_width_popup("<b>Dobanovački zabran</b><br>October 2024"), 
    icon=folium.Icon(color=color_forest, icon="dove", prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.68962, 20.51490], 
    popup=dynamic_width_popup("<b>Avala</b><br>December 2024"), 
    icon=folium.Icon(color=color_mountain, icon=icon_mountain, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.72089, 20.51674], 
    popup=dynamic_width_popup("<b>Stepin Lug</b><br>December 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.73524, 20.50367], 
    popup=dynamic_width_popup("<b>Torlačka šuma</b><br>December 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.73015, 20.49012], 
    popup=dynamic_width_popup("<b>Jajinačka šuma</b><br>December 2024"), 
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.69251, 20.40320], 
    popup=dynamic_width_popup("<b>Sremačka šuma</b><br>April 2025"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.86293, 20.49304],
    popup=dynamic_width_popup("<b>Veliko Blato</b><br>May 2025"),
    icon=folium.Icon(color=color_water, icon=icon_water, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.72717, 20.68794],
    popup=dynamic_width_popup("<b>Ivanovačka ada</b><br>July 2025"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [44.68125, 20.55128],
    popup=dynamic_width_popup("<b>Karagača</b><br>October 2025"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [45.22276, 19.60329],
    popup=dynamic_width_popup("<b>Begečka jama</b><br>October 2025"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [45.16906, 19.91733],
    popup=dynamic_width_popup("<b>Stražilovo</b><br>May 2026"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)

folium.Marker(
    [45.16801, 20.01008],
    popup=dynamic_width_popup("<b>Čortanovačka šuma</b><br>May 2026"),
    icon=folium.Icon(color=color_forest, icon=icon_forest, prefix="fa"), z_index_offset=z_index_nature
).add_to(m)




In [30]:
m.save("srbija.html")